# Clase 13 · Notebook 01
# Escalado, diagnóstico de ajuste y early stopping

**Introducción al Deep Learning — Módulo 2, Unidad 3: Regresión (Parte II)**

Tres prácticas que mejoran un modelo de regresión:

1. **Escalar** las variables y ver su efecto en el entrenamiento.
2. **Diagnosticar** el ajuste con las curvas de aprendizaje (overfitting).
3. Aplicar **early stopping**.

Usamos el dataset *diabetes*.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. El efecto del escalado

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
tf.random.set_seed(42); np.random.seed(42)

d = load_diabetes()
# Desescalamos a propósito una versión "cruda" (multiplicamos para exagerar las escalas)
X_raw = d.data * np.array([1, 1, 1000, 500, 1, 1, 1, 1, 100, 1])
y = d.target.astype("float32")
Xtr_r, Xte_r, ytr, yte = train_test_split(X_raw, y, test_size=0.2, random_state=42)

sc = StandardScaler().fit(Xtr_r)               # ajustamos SOLO con el train
Xtr_s, Xte_s = sc.transform(Xtr_r), sc.transform(Xte_r)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
def crear():
    tf.random.set_seed(42)
    m = Sequential([Input(shape=(10,)), Dense(32, activation="relu"), Dense(1)])
    m.compile(optimizer="adam", loss="mse"); return m

h_raw = crear().fit(Xtr_r.astype("float32"), ytr, epochs=80, verbose=0, validation_split=0.1)
h_sc  = crear().fit(Xtr_s.astype("float32"), ytr, epochs=80, verbose=0, validation_split=0.1)

plt.figure(figsize=(7, 4))
plt.plot(h_raw.history["loss"], label="sin escalar", color="#4355FF")
plt.plot(h_sc.history["loss"],  label="escalado",   color="#FF647E")
plt.title("Pérdida (MSE) con y sin escalado"); plt.xlabel("Época"); plt.ylabel("MSE")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("MSE final sin escalar: %.0f" % h_raw.history["loss"][-1])
print("MSE final escalado   : %.0f" % h_sc.history["loss"][-1])

El modelo con datos **escalados** entrena de forma mucho más estable y baja antes la pérdida.
Nota: el escalador se ajusta **solo con el entrenamiento** y se aplica al test (evita fugas de información).

## 2. Diagnosticar el ajuste: overfitting

Con una red grande y pocos datos, la pérdida de **entrenamiento** baja mucho pero la de **validación** se
estanca o sube: es **overfitting**.

In [ ]:
Xtr2, Xte2, ytr2, yte2 = train_test_split(StandardScaler().fit_transform(d.data), y,
                                            test_size=0.2, random_state=42)
tf.random.set_seed(42)
grande = Sequential([Input(shape=(10,)),
                     Dense(128, activation="relu"),
                     Dense(128, activation="relu"),
                     Dense(1)])
grande.compile(optimizer="adam", loss="mse")
h = grande.fit(Xtr2, ytr2, epochs=300, batch_size=8, validation_split=0.3, verbose=0)

plt.figure(figsize=(7, 4))
plt.plot(h.history["loss"], label="entrenamiento", color="#000F9F")
plt.plot(h.history["val_loss"], label="validación", color="#FF647E")
plt.title("Curvas de aprendizaje (overfitting)"); plt.xlabel("Época"); plt.ylabel("MSE")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 3. Early stopping

Detenemos el entrenamiento cuando la validación deja de mejorar y restauramos los mejores pesos.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import r2_score

es = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
tf.random.set_seed(42)
m = Sequential([Input(shape=(10,)), Dense(128, activation="relu"), Dense(128, activation="relu"), Dense(1)])
m.compile(optimizer="adam", loss="mse")
h_es = m.fit(Xtr2, ytr2, epochs=400, batch_size=8, validation_split=0.3, verbose=0, callbacks=[es])

print("Épocas ejecutadas:", len(h_es.history["loss"]), "(se detuvo solo)")
print("R2 en test: %.3f" % r2_score(yte2, m.predict(Xte2, verbose=0).ravel()))

## Resumen

- **Escalar** las variables acelera y estabiliza el entrenamiento; ajusta el escalador solo con el train.
- Las **curvas de aprendizaje** revelan el overfitting (validación que empeora).
- **Early stopping** para a tiempo y conserva los mejores pesos.

➡️ En el **Notebook 02** estimaremos el error de forma robusta con validación cruzada y analizaremos los residuos.